Unduh data korpus Wikipedia Bahasa Indonesia melalui link berikut https://dumps.wikimedia.org/idwiki/latest/. Cari file dengan nama “idwiki-latest-pages-articles.xml.bz2”. Data korpus setiap waktunya mengalami peningkatan. Sekitar agustus 2019 terdapat 406855 kosa kata dengan ukuran 512 MB.

Data korpus tersebut masih menggunakan format XML sehingga perlu di olah ke dalam bentuk teks. library Gensim sudah menyediakan teknik pra poses ini. Untuk mengkonversi format tersebut dapat menggunakan kode berikut:

In [46]:
import pandas as pd

# Baca beberapa baris pertama dari file CSV
data = pd.read_csv("bbc_data.csv")
print(data.columns)  # Menampilkan nama-nama kolom dalam file CSV


Index(['data', 'labels'], dtype='object')


### pembetulan yang eror dibawahnya

In [47]:
import pandas as pd
import logging

# Inisialisasi logging
logging.basicConfig(format='%(asctime)s: %(levelname)s: %(message)s')
logging.root.setLevel(level=logging.INFO)
logger = logging.getLogger()

namaFileInput = "bbc_data.csv"
namaFileOutput = "wiki.id.text"

# Baca file CSV
data = pd.read_csv(namaFileInput)

# Menggunakan kolom 'data' sebagai sumber teks
if 'data' in data.columns:
    with open(namaFileOutput, 'w', encoding='utf-8') as output:
        for i, text in enumerate(data['data']):
            output.write(text + '\n')
            if (i + 1) % 10000 == 0:
                logger.info("Saved " + str(i + 1) + " articles")
    
    logger.info("Finished Saving " + str(i + 1) + " articles")
else:
    print("Kolom 'data' tidak ditemukan dalam file CSV.")


2024-10-27 15:47:22,897: INFO: Finished Saving 2225 articles


### error huhuhu

In [48]:
from __future__ import print_function
 
# Ignore warnings dari gensim
import warnings
warnings.filterwarnings(action='ignore', category=UserWarning, module='gensim')
 
import logging
import os.path
import sys
 
from gensim.corpora import WikiCorpus
 
program = os.path.basename(sys.argv[0])
logger = logging.getLogger(program)
 
logging.basicConfig(format='%(asctime)s: %(levelname)s: %(message)s')
logging.root.setLevel(level=logging.INFO)
logger.info("running %s" % ' '.join(sys.argv))
 
#namaFileInput = "idwiki-latest-pages-articles.xml.bz2"
namaFileInput = "bbc_data.csv"
namaFileOutput = "wiki.id.text"
 
space = " "
i = 0
 
# Write file ke variabel namaFileOutput encoder utf-8
output = open(namaFileOutput, 'w', encoding='utf-8')
 
# lower=False: huruf kecil dan besar dibedakan
wiki = WikiCorpus(namaFileInput, lemmatize=None, dictionary={}, lower=False)
for text in wiki.get_texts():
    output.write(' '.join(text) + '\n')
    i = i + 1
    if i % 10000 == 0:
        logger.info("Saved " + str(i) + " articles")
 
output.close()
logger.info("Finished Saved " + str(i) + " articles")

2024-10-27 15:47:22,910: INFO: running d:\stki\env\Lib\site-packages\ipykernel_launcher.py --f="c:\Users\gabriella fani\AppData\Roaming\jupyter\runtime\kernel-v3f352e409e05d1d83a2b86164c3797ffd3538edac.json"


OSError: Invalid data stream

Dari kode di atas akan menghasilkan output file dari rangkaian beberapa artikel, di mana satu barisnya mewakili satu artikel.

Setelah data berhasil terkonversi, selanjutnya dilakukan training Word2Vec menggunakan kode berikut:

### ngebentuk modelnya

In [22]:
import multiprocessing
import logging
import os.path
import sys
import multiprocessing
from gensim.models import Word2Vec
from gensim.models.word2vec import LineSentence
 
program = os.path.basename(sys.argv[0])
logger = logging.getLogger(program)
 
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s')
logging.root.setLevel(level=logging.INFO)
logger.info("running %s" % ' '.join(sys.argv))
 
namaFileInput = "wiki.id.text"
namaFileOutput = "w2vec_wiki_id300_0.txt"
 
 
# size 300 = 300 dimensi vektor, window 10 = 10 pengaruh kata disekitarnya, min_count 5 = kata-kata yang muncul < 5 kali akan dikeluarkan dari kosakata dan diabaikan selama pelatihan, sg 0 = cbow / sg 1 = skip gram 
model = Word2Vec(LineSentence(namaFileInput), vector_size=300, window=10, min_count=5, sg=0, workers=multiprocessing.cpu_count())
 
# trim unneeded model memory = use (much) less RAM
model.init_sims(replace=True)
model.wv.save_word2vec_format(namaFileOutput, binary=False)

2024-10-27 14:37:47,293: INFO: running d:\stki\env\Lib\site-packages\ipykernel_launcher.py --f="c:\Users\gabriella fani\AppData\Roaming\jupyter\runtime\kernel-v3f352e409e05d1d83a2b86164c3797ffd3538edac.json"
2024-10-27 14:37:47,297: INFO: collecting all words and their counts
2024-10-27 14:37:47,298: INFO: PROGRESS: at sentence #0, processed 0 words, keeping 0 word types
2024-10-27 14:37:47,510: INFO: collected 64151 word types from a corpus of 854770 raw words and 2225 sentences
2024-10-27 14:37:47,510: INFO: Creating a fresh vocabulary
2024-10-27 14:37:47,564: INFO: Word2Vec lifecycle event {'msg': 'effective_min_count=5 retains 14313 unique words (22.31% of original 64151, drops 49838)', 'datetime': '2024-10-27T14:37:47.564805', 'gensim': '4.3.3', 'python': '3.11.3 (tags/v3.11.3:f3909b8, Apr  4 2023, 23:49:59) [MSC v.1934 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.22631-SP0', 'event': 'prepare_vocab'}
2024-10-27 14:37:47,565: INFO: Word2Vec lifecycle event {'msg': 'effective_min

Terdapat beberapa parameter diantaranya:

<b>vector_size</b>: Ukuran dimensi vector yang mewakili setiap token atau kata. Jika memiliki data yang terbatas, nilai size yang digunakan sebaiknya lebih kecil karena akan mempengaruhi kata unik di sekitarnya. Namun, jika memiliki banyak data maka dapat bereksperimen dengan berbagai ukuran.

<b>window</b>: Jarak maksimum antara kata target dengan kata di sekitarnya. Jika nilai window yang digunakan besar, maka terdapat banyak kata terkait disekitarnya (baik di posisi kiri dan kanan dari kata konteks). Secara teori, jika window lebih kecil akan memberikan istilah yang lebih spesifik terhadap suatu konteks kata.

<b>min_count</b>: Frekuensi minimal jumlah kata. Model akan mengabaikan kata-kata yang tidak memenuhi nilai min_count. Kata yang jarang muncul biasanya tidak terlalu penting, jadi lebih baik untuk dihilangkan. Parameter ini mungkin lebih berpengaruh pada effisiensi penggunaan memory dan ukuran file model.

<b>sg</b>: 0 untuk arsitektur Word2Vec CBOW dan 1 untuk arsitektur Word2Vec Skip-gram.

<b>workers</b>: Berapa banyak threads yang digunakan untuk melakukan multiprocessing.

Model yang sudah disimpan dapat dilakukan skenario penggunaannya sebagaimana pada kode di bawah ini untuk melihat kedekatan hubungan antara kata.

In [25]:
import gensim

namaFileModel = "w2vec_wiki_id300_0.txt"

# Muat model dengan format teks
model = gensim.models.KeyedVectors.load_word2vec_format(namaFileModel, binary=False)


2024-10-27 14:41:30,864: INFO: loading projection weights from w2vec_wiki_id300_0.txt
2024-10-27 14:41:33,411: INFO: KeyedVectors lifecycle event {'msg': 'loaded (14313, 300) matrix of type float32 from w2vec_wiki_id300_0.txt', 'binary': False, 'encoding': 'utf8', 'datetime': '2024-10-27T14:41:33.411511', 'gensim': '4.3.3', 'python': '3.11.3 (tags/v3.11.3:f3909b8, Apr  4 2023, 23:49:59) [MSC v.1934 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.22631-SP0', 'event': 'load_word2vec_format'}


### print 10 kata teratas dari dataset

In [29]:
import pandas as pd

# Baca dataset dari file CSV
namaFileInput = "bbc_data.csv"
data = pd.read_csv(namaFileInput)

# Tampilkan 10 kata pertama dari kolom tertentu, misalnya kolom 'data'
print(data['data'].head(10))


0    Musicians to tackle US red tape  Musicians gro...
1    U2s desire to be number one  U2, who have won ...
2    Rocker Doherty in on-stage fight  Rock singer ...
3    Snicket tops US box office chart  The film ada...
4    Oceans Twelve raids box office  Oceans Twelve,...
5    Landmark movies of 2004 hailed  US film profes...
6    Pete Doherty misses bail deadline  Singer Pete...
7    Fockers retain film chart crown  Comedy Meet T...
8    Top gig award for Scissor Sisters  New York ba...
9    Johnny Depp: The acting outlaw  Johnny Depp, w...
Name: data, dtype: object


In [33]:
import gensim

# Memuat model
namaFileModel = "w2vec_wiki_id300_0.txt"
model = gensim.models.KeyedVectors.load_word2vec_format(namaFileModel, binary=False)

# Fungsi untuk mendapatkan dan mencetak hasil most_similar_cosmul
def get_similar_words(model, positive, negative):
    # Memeriksa apakah semua kata ada dalam vocabulary
    if all(word in model for word in positive + negative):
        sim = model.most_similar_cosmul(positive=positive, negative=negative)
        print(f"{' + '.join(positive)} - {' - '.join(negative)}: {sim}")
    else:
        missing_words = [word for word in positive + negative if word not in model]
        print(f"Kata yang tidak ditemukan dalam vocabulary: {', '.join(missing_words)}")

2024-10-27 15:16:41,020: INFO: loading projection weights from w2vec_wiki_id300_0.txt
2024-10-27 15:16:43,478: INFO: KeyedVectors lifecycle event {'msg': 'loaded (14313, 300) matrix of type float32 from w2vec_wiki_id300_0.txt', 'binary': False, 'encoding': 'utf8', 'datetime': '2024-10-27T15:16:43.478941', 'gensim': '4.3.3', 'python': '3.11.3 (tags/v3.11.3:f3909b8, Apr  4 2023, 23:49:59) [MSC v.1934 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.22631-SP0', 'event': 'load_word2vec_format'}


In [40]:
import gensim
import gensim.models.keyedvectors as word2vec
model = gensim.models.Word2Vec.load(namaFileModel)
model = word2vec.KeyedVectors.load_word2vec_format(namaFileModel, binary=True)

namaFileModel = "w2vec_wiki_id300_0.txt"

sim = model.wv.most_similar("Musicians")
print("10 kata terdekat dari Musicians:{}".format(sim))
sim = model.wv.most_similar("movies")
print("10 kata terdekat dari movies:{}".format(sim))
 
#sim = model.wv.most_similar_cosmul(positive=['music', 'rap'], negative=['band'])
#print("band-rap, music-?: {}".format(sim))
#sim = model.wv.most_similar_cosmul(positive=['film', 'festival'], negative=['hollywood'])
#print("hollywood-festival  , mobilfilm-?: {}".format(sim))

2024-10-27 15:33:28,836: INFO: loading Word2Vec object from w2vec_wiki_id300_0.txt


UnpicklingError: could not find MARK

In [45]:
import gensim

# Memuat model dalam format teks
namaFileModel = "w2vec_wiki_id300_0.txt"
model = gensim.models.KeyedVectors.load_word2vec_format(namaFileModel, binary=False)

# Memeriksa kosakata
print("Kosakata dalam model:", list(model.key_to_index.keys())[:20])  # Menampilkan 20 kata pertama

# Mengambil kata terdekat dari "Musicians"
sim = model.most_similar("Musicians")
print("10 kata terdekat dari 'Musicians': {}".format(sim))

# Mengambil kata terdekat dari "movies"
sim = model.most_similar("movies")
print("10 kata terdekat dari 'movies': {}".format(sim))

# Memeriksa dan menghitung kesamaan
words_to_check = ["musicians", "movies"]

# Cek apakah kata ada dalam model sebelum menghitung kesamaan
for i in range(len(words_to_check)):
    for j in range(i + 1, len(words_to_check)):
        word1 = words_to_check[i]
        word2 = words_to_check[j]
        if word1 in model.key_to_index and word2 in model.key_to_index:
            print(f"Kedekatan '{word1}' dan '{word2}': {model.similarity(word1, word2)}")
        else:
            if word1 not in model.key_to_index:
                print(f"Kata '{word1}' tidak ditemukan dalam model.")
            if word2 not in model.key_to_index:
                print(f"Kata '{word2}' tidak ditemukan dalam model.")

# Menggunakan most_similar_cosmul
sim = model.most_similar_cosmul(positive=['music', 'rap'], negative=['band'])
print("band-rap, music-?: {}".format(sim))

#sim = model.most_similar_cosmul(positive=['film', 'festival'], negative=['hollywood'])
#print("hollywood-festival, film-?: {}".format(sim))


2024-10-27 15:40:03,853: INFO: loading projection weights from w2vec_wiki_id300_0.txt
2024-10-27 15:40:06,917: INFO: KeyedVectors lifecycle event {'msg': 'loaded (14313, 300) matrix of type float32 from w2vec_wiki_id300_0.txt', 'binary': False, 'encoding': 'utf8', 'datetime': '2024-10-27T15:40:06.917654', 'gensim': '4.3.3', 'python': '3.11.3 (tags/v3.11.3:f3909b8, Apr  4 2023, 23:49:59) [MSC v.1934 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.22631-SP0', 'event': 'load_word2vec_format'}


Kosakata dalam model: ['the', 'to', 'of', 'and', 'a', 'in', 'for', 'is', 'that', 'The', 'on', 'was', 'be', 'with', 'has', 'said', 'it', 'have', 'as', 'will']
10 kata terdekat dari 'Musicians': [('Leisure', 0.7995740175247192), ('removal', 0.7940601110458374), ('Coast', 0.7683220505714417), ('Jacques', 0.7615554928779602), ('Web', 0.7610881924629211), ('proposing', 0.7601100206375122), ('Anne', 0.7572643160820007), ('Booker', 0.755321741104126), ('Science', 0.7514492869377136), ('Forum', 0.7498074173927307)]
10 kata terdekat dari 'movies': [('networks.', 0.954998254776001), ('checks', 0.9532687067985535), ('250', 0.9513975977897644), ('programmes', 0.9474685788154602), ('homes.', 0.9450694918632507), ('tasks', 0.9417365789413452), ('supplies', 0.9413403272628784), ('storage', 0.9401395320892334), ('swap', 0.9400225877761841), ('schools', 0.9399439692497253)]
Kedekatan 'musicians' dan 'movies': 0.8099606037139893
band-rap, music-?: [('video', 1.0518686771392822), ('technology', 1.0330154

In [24]:
import gensim
# import gensim.models.keyedvectors as word2vec
# model = gensim.models.Word2Vec.load(namaFileModel)
# model = word2vec.KeyedVectors.load_word2vec_format(namaFileModel, binary=True)

namaFileModel = "w2vec_wiki_id300_0.txt"

sim = model.wv.most_similar("Jakarta")
print("10 kata terdekat dari Jakarta:{}".format(sim))
sim = model.wv.most_similar("Bandung")
print("10 kata terdekat dari Bandung:{}".format(sim))
 
sim = model.wv.similarity("Yogyakarta", "Surakarta")
print("Kedekatan Yogyakarta-Surakarta: {}".format(sim))
sim = model.wv.similarity("Yogyakarta", "Semarang")
print("Kedekatan Yogyakarta-Semarang: {}".format(sim))
 
sim = model.wv.most_similar_cosmul(positive=['minuman', 'rendang'], negative=['makanan'])
print("makanan-rendang, minuman-?: {}".format(sim))
sim = model.wv.most_similar_cosmul(positive=['mobil', 'honda'], negative=['motor'])
print("motor-honda, mobil-?: {}".format(sim))

10 kata terdekat dari Jakarta:[('topping', 0.9829132556915283), ('maker', 0.9807416200637817), ('Indian', 0.979942262172699), ('2.2%', 0.9798845052719116), ('completed', 0.9790072441101074), ('blamed', 0.9762697219848633), ('Motion', 0.9753372669219971), ('Ethiopia', 0.9752442836761475), ('healthy', 0.9750059247016907), ('pre-tax', 0.974707841873169)]


KeyError: "Key 'Bandung' not present in vocabulary"